# Analyse de Sentiments : BERT <-> RoBERTa Cross-Attention Fusion

**Pipeline :** `1. Data` -> `2. EDA` -> `3. Prétraitement` -> `4. Transformations` -> `5. ML` -> `6. EVAL` -> `7. Déploiement` (+ Cross-Dataset Testing)

# 1. Data
Chargement du jeu de données Google Reviews Malaysia (`Review` et `Rating`), vérification de l'intégrité des colonnes et sauvegarde de `cleaned_reviews.csv`.

In [ ]:
import os
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/choonkhonng/malaysia-restaurant-review-datasets/Malaysia Restaurant Review Datasets/data_cleaned/GoogleReview_data_cleaned.csv"
TEXT_COL = "Review"
LABEL_COL = "Rating"
SEED = 42


class ReviewDataSource:
    def __init__(self, path, text_col, label_col):
        self.path = path
        self.text_col = text_col
        self.label_col = label_col

    def load(self):
        frame = pd.read_csv(self.path)
        missing_cols = [c for c in (self.text_col, self.label_col) if c not in frame.columns]
        if missing_cols:
            raise ValueError(
                f"Colonnes introuvables : {missing_cols}. Colonnes disponibles : {list(frame.columns)}"
            )
        return frame[[self.text_col, self.label_col]].rename(
            columns={self.text_col: "text", self.label_col: "rating"}
        )


raw_df = ReviewDataSource(DATA_PATH, TEXT_COL, LABEL_COL).load()
os.makedirs("/kaggle/working", exist_ok=True)
raw_df.to_csv("/kaggle/working/cleaned_reviews.csv", index=False)

print("Shape :", raw_df.shape)
print("\nTypes :\n", raw_df.dtypes)
print("\nValeurs manquantes :\n", raw_df.isnull().sum())
raw_df.head()

# 2. EDA
Analyse exploratoire : volumétrie, doublons, distribution des notes/polarités et longueur des avis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


class ReviewExplorer:
    def __init__(self, frame):
        self.frame = frame.dropna().copy()
        self.frame["text"] = self.frame["text"].astype(str).str.strip()
        self.frame = self.frame[self.frame["text"].str.len() > 0]
        self.frame["word_count"] = self.frame["text"].str.split().str.len()
        self.frame["sentiment"] = self.frame["rating"].apply(
            lambda r: "Positive" if r >= 4 else ("Neutral" if r == 3 else "Negative")
        )

    def summary(self):
        dedup = self.frame.drop_duplicates(subset=["text"])
        print("Avis valides (non vides) :", len(self.frame))
        print("Textes dupliqués :", self.frame["text"].duplicated().sum())
        print("Avis uniques après déduplication :", len(dedup))
        print("\nDistribution des polarités (avis uniques) :")
        print(dedup["sentiment"].value_counts())
        print("\nStatistiques du nombre de mots (avis uniques) :")
        print(dedup["word_count"].describe())

    def plot_rating_distribution(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        self.frame["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C72B0")
        axes[0].set_title("Distribution des notes (Rating)")
        axes[0].set_xlabel("Note")
        axes[0].set_ylabel("Nombre d'avis")

        order = ["Negative", "Neutral", "Positive"]
        colors = ["#C44E52", "#DD8452", "#55A868"]
        dedup = self.frame.drop_duplicates(subset=["text"])
        dedup["sentiment"].value_counts().reindex(order).plot(kind="bar", ax=axes[1], color=colors)
        axes[1].set_title("Distribution des classes (avis uniques)")
        axes[1].set_xlabel("Polarité")
        axes[1].set_ylabel("Nombre d'avis")
        plt.tight_layout()
        plt.show()

    def plot_length_distribution(self):
        plt.figure(figsize=(8, 4))
        sns.histplot(self.frame["word_count"].clip(upper=300), bins=50, color="#4C72B0")
        plt.title("Distribution du nombre de mots par avis (tronqué à 300)")
        plt.xlabel("Nombre de mots")
        plt.ylabel("Fréquence")
        plt.tight_layout()
        plt.show()

    def plot_length_by_rating(self):
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=self.frame, x="rating", y="word_count")
        plt.ylim(0, 300)
        plt.title("Longueur des avis par note")
        plt.xlabel("Note")
        plt.ylabel("Nombre de mots")
        plt.tight_layout()
        plt.show()


explorer = ReviewExplorer(raw_df)
explorer.summary()
explorer.plot_rating_distribution()
explorer.plot_length_distribution()
explorer.plot_length_by_rating()

### Interprétation des graphiques

* **Déséquilibre marqué des classes :** la distribution met en évidence une forte prédominance des avis positifs (**156 390**) face aux avis négatifs (**20 405**), les avis neutres (**22 376**) étant également minoritaires. Entraîner un modèle directement sur ces données brutes introduirait un biais important en faveur de la classe majoritaire et dégraderait les performances (notamment le rappel) sur les classes sous-représentées.
* **Justification du sous-échantillonnage :** afin de corriger ce déséquilibre et d’assurer un apprentissage équitable entre les trois polarités (*Negative*, *Neutral* et *Positive*), un sous-échantillonnage équilibré à **5 000 avis par classe** (soit **15 000** avis au total) est appliqué à l’étape suivante.

# 3. Prétraitement
Nettoyage, déduplication, mapping des labels (`Negative`: 0, `Neutral`: 1, `Positive`: 2), sous-échantillonnage équilibré (5 000/classe) et découpage stratifié **70 % Train / 10 % Validation / 20 % Test**.

In [ ]:
from sklearn.model_selection import train_test_split


class ReviewPreprocessor:
    LABELS = {"Negative": 0, "Neutral": 1, "Positive": 2}

    def __init__(self, per_class=5000, test_size=0.20, val_size=0.10, seed=42):
        self.per_class = per_class
        self.test_size = test_size
        self.val_size = val_size
        self.seed = seed

    def clean(self, frame):
        frame = frame.dropna().copy()
        frame["text"] = frame["text"].astype(str).str.strip()
        frame = frame[frame["text"].str.len() > 0]
        return frame.drop_duplicates(subset=["text"])

    @staticmethod
    def to_sentiment(rating):
        if rating >= 4:
            return "Positive"
        if rating == 3:
            return "Neutral"
        return "Negative"

    def balance(self, frame):
        parts = [
            group.sample(n=min(self.per_class, len(group)), random_state=self.seed)
            for _, group in frame.groupby("sentiment")
        ]
        return pd.concat(parts).sample(frac=1, random_state=self.seed).reset_index(drop=True)

    def split(self, frame):
        train_val, test = train_test_split(
            frame, test_size=self.test_size, stratify=frame["label"], random_state=self.seed
        )
        val_ratio = self.val_size / (1.0 - self.test_size)
        train, val = train_test_split(
            train_val, test_size=val_ratio, stratify=train_val["label"], random_state=self.seed
        )
        return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

    def run(self, frame):
        frame = self.clean(frame)
        frame["sentiment"] = frame["rating"].apply(self.to_sentiment)
        frame = self.balance(frame)
        frame["label"] = frame["sentiment"].map(self.LABELS)
        return self.split(frame)


train_df, val_df, test_df = ReviewPreprocessor(per_class=5000, seed=SEED).run(raw_df)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} : {part.shape} | {part['sentiment'].value_counts().to_dict()}")
    part.to_csv(f"/kaggle/working/{name}.csv", index=False)

# 4. Transformations
Double tokenisation (`bert-base-uncased` et `roberta-base`) et construction des `DataLoader` PyTorch.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

BERT_NAME = "bert-base-uncased"
ROBERTA_NAME = "roberta-base"
MAX_LEN = 256
BATCH_SIZE = 16

bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)
roberta_tok = AutoTokenizer.from_pretrained(ROBERTA_NAME)


class DualTokenizerDataset(Dataset):
    def __init__(self, frame, first_tokenizer, second_tokenizer, max_len):
        self.texts = frame["text"].astype(str).tolist()
        self.labels = frame["label"].tolist()
        self.first_tokenizer = first_tokenizer
        self.second_tokenizer = second_tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def encode(self, tokenizer, text):
        return tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

    def __getitem__(self, index):
        text = self.texts[index]
        first = self.encode(self.first_tokenizer, text)
        second = self.encode(self.second_tokenizer, text)
        return {
            "bert_ids": first["input_ids"].squeeze(0),
            "bert_mask": first["attention_mask"].squeeze(0),
            "roberta_ids": second["input_ids"].squeeze(0),
            "roberta_mask": second["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[index], dtype=torch.long),
        }


def build_loader(frame, shuffle):
    dataset = DualTokenizerDataset(frame, bert_tok, roberta_tok, MAX_LEN)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader = build_loader(train_df, shuffle=True)
val_loader = build_loader(val_df, shuffle=False)
test_loader = build_loader(test_df, shuffle=False)

print(f"Batches -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

# 5. ML — BERT <-> RoBERTa Cross-Attention Fusion (Case 5)
Entraînement en modes `frozen` et `finetuned` avec sélection du meilleur checkpoint sur le jeu de **validation**, puis évaluation unique sur le jeu de **test**.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ===================== CONFIG =====================
NUM_CLASSES = 3
NUM_HEADS = 8
EPOCHS_FROZEN = 4
EPOCHS_FINETUNED = 3
LR_FROZEN = 5e-4
LR_FINETUNED = 2e-5
WARMUP_RATIO = 0.1
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TEXT_COL, LABEL_COL = "text", "label"


# ===================== Model =====================
class CrossAttentionFusionModel(nn.Module):
    def __init__(self, num_classes, freeze=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BERT_NAME)
        self.roberta = AutoModel.from_pretrained(ROBERTA_NAME)
        hidden = self.bert.config.hidden_size  # 768
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=NUM_HEADS, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden, num_classes)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
            for p in self.roberta.parameters():
                p.requires_grad = False

    def forward(self, bert_ids, bert_mask, roberta_ids, roberta_mask):
        bert_hidden = self.bert(input_ids=bert_ids, attention_mask=bert_mask).last_hidden_state
        roberta_out = self.roberta(input_ids=roberta_ids, attention_mask=roberta_mask).last_hidden_state
        roberta_cls = roberta_out[:, 0:1, :]

        key_padding_mask = (bert_mask == 0)
        attn_out, _ = self.cross_attn(
            query=roberta_cls,
            key=bert_hidden,
            value=bert_hidden,
            key_padding_mask=key_padding_mask,
        )
        fused = self.norm(attn_out.squeeze(1) + roberta_out[:, 0, :])
        return self.classifier(self.dropout(fused))


# ===================== Train / Eval =====================
def train_epoch(model, loader, opt, crit, sched):
    model.train()
    total = 0.0
    for batch in loader:
        bi, bm = batch["bert_ids"].to(device), batch["bert_mask"].to(device)
        ri, rm = batch["roberta_ids"].to(device), batch["roberta_mask"].to(device)
        y = batch["label"].to(device)
        opt.zero_grad()
        logits = model(bi, bm, ri, rm)
        loss = crit(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if sched:
            sched.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader, crit=None):
    model.eval()
    preds, labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            bi, bm = batch["bert_ids"].to(device), batch["bert_mask"].to(device)
            ri, rm = batch["roberta_ids"].to(device), batch["roberta_mask"].to(device)
            y = batch["label"].to(device)
            logits = model(bi, bm, ri, rm)
            if crit is not None:
                total_loss += crit(logits, y).item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labels.extend(y.cpu().numpy())
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    avg_loss = (total_loss / len(loader)) if crit is not None else None
    return avg_loss, acc, p, r, f1


def run(mode):
    freeze = mode == "frozen"
    epochs = EPOCHS_FROZEN if freeze else EPOCHS_FINETUNED
    lr = LR_FROZEN if freeze else LR_FINETUNED
    print(f"\n{'='*60}\nCase 5: BERT<->RoBERTa Cross-Attention | mode={mode} | epochs={epochs} | lr={lr}\n{'='*60}")

    model = CrossAttentionFusionModel(NUM_CLASSES, freeze=freeze).to(device)
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    crit = nn.CrossEntropyLoss()
    total_steps = len(train_loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total_steps), total_steps)

    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_epoch = 1
    best_state = None

    start = time.time()
    for ep in range(epochs):
        train_loss = train_epoch(model, train_loader, opt, crit, sched)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, crit)
        print(
            f"Epoch {ep+1}/{epochs} - train_loss: {train_loss:.4f} - "
            f"val_loss: {val_loss:.4f} - val_acc: {val_acc*100:.2f}%"
        )
        history["epoch"].append(ep + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc * 100)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = ep + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restauration du meilleur checkpoint sélectionné sur validation avant l'unique passe sur test
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"Best checkpoint selected at epoch {best_epoch} (val_acc={best_val_acc*100:.2f}%)")

    _, acc, p, r, f1 = evaluate(model, test_loader)
    duration = time.time() - start
    print(f"Final Test -> Acc:{acc*100:.2f}% Prec:{p*100:.2f}% Rec:{r*100:.2f}% F1:{f1*100:.2f}% Time:{duration:.1f}s")

    summary = {
        "model": "BERT<->RoBERTa (Cross-Attn)",
        "mode": mode,
        "best_epoch": best_epoch,
        "val_acc": round(best_val_acc * 100, 2),
        "accuracy": round(acc * 100, 2),
        "precision": round(p * 100, 2),
        "recall": round(r * 100, 2),
        "f1": round(f1 * 100, 2),
        "train_time_sec": round(duration, 1),
    }
    return summary, history, model


# ===================== Run both modes =====================
results = []
histories = {}
trained_models = {}
for m in ["frozen", "finetuned"]:
    res, hist, trained_model = run(m)
    results.append(res)
    histories[m] = hist
    trained_models[m] = trained_model

results_df = pd.DataFrame(results)
print("\n" + "=" * 85)
print("CASE 5 RESULTS - BERT <-> RoBERTa Cross-Attention Fusion")
print("=" * 85)
print(results_df.to_string(index=False))

# ===================== Graphs (inline only) =====================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for m, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=f"{m} (train)")
    axes[0].plot(hist["epoch"], hist["val_loss"], marker="s", linestyle="--", label=f"{m} (val)")
    axes[1].plot(hist["epoch"], hist["val_acc"], marker="o", label=m)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Case 5 - Train & Validation Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Accuracy (%)")
axes[1].set_title("Case 5 - Validation Accuracy Curve")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
bars = plt.bar(results_df["mode"], results_df["accuracy"], color=["#A9D18E", "#548235"])
for bar, val in zip(bars, results_df["accuracy"]):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.5, f"{val:.2f}%", ha="center", fontweight="bold")
plt.ylabel("Test Accuracy (%)")
plt.title("Case 5 - BERT<->RoBERTa Cross-Attention Test Accuracy")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

metrics = ["precision", "recall", "f1"]
x = np.arange(len(results_df["mode"]))
width = 0.25
plt.figure(figsize=(8, 5))
for i, metric in enumerate(metrics):
    plt.bar(x + i * width, results_df[metric], width, label=metric.capitalize())
plt.xticks(x + width, results_df["mode"])
plt.ylabel("Score (%)")
plt.title("Case 5 - Precision / Recall / F1-score")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
bars = plt.bar(results_df["mode"], results_df["train_time_sec"], color="#70AD47")
for bar, val in zip(bars, results_df["train_time_sec"]):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 3, f"{val:.0f}s", ha="center", fontweight="bold")
plt.ylabel("Training Time (seconds)")
plt.title("Case 5 - Execution Time")
plt.tight_layout()
plt.show()

# 6. EVAL — Évaluation Statistique, Sondes & Tests Unitaires
* **A. Évaluation statistique avancée** : rapport de classification, matrice de confusion, intervalles de confiance bootstrap (95 %), significativité vs baseline majoritaire, analyse d'erreurs et sauvegarde de `metrics.json`.
* **B. Tests sur phrases sondes** : sarcasme, négation, sentiment mixte, texte court, code-mixing (Manglish / Hinglish).
* **C. Tests unitaires du pipeline** : formes des tenseurs, masques binaires, déterminisme, absence de fuite entre batchs et troncature.

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
)

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
label2id = {v: k for k, v in id2label.items()}

best_model = trained_models["finetuned"]
best_model.eval()


# ================================================================================
# A. ADVANCED STATISTICAL EVALUATION
# ================================================================================
def get_all_predictions(model, loader):
    """Run inference over a loader once, return preds, true labels, and softmax probs."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            bi, bm = batch["bert_ids"].to(device), batch["bert_mask"].to(device)
            ri, rm = batch["roberta_ids"].to(device), batch["roberta_mask"].to(device)
            y = batch["label"].to(device)
            logits = model(bi, bm, ri, rm)
            probs = torch.softmax(logits, dim=1)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=id2label.values(), yticklabels=id2label.values(), ax=axes[0]
    )
    axes[0].set_title(title + " (counts)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=id2label.values(), yticklabels=id2label.values(), ax=axes[1]
    )
    axes[1].set_title(title + " (row-normalized)")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")
    plt.tight_layout()
    plt.show()
    return cm


def bootstrap_ci(y_true, y_pred, metric_fn=accuracy_score, n_boot=1000, ci=95, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    lo = np.percentile(scores, (100 - ci) / 2)
    hi = np.percentile(scores, 100 - (100 - ci) / 2)
    return np.mean(scores), lo, hi


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def error_analysis(test_df, y_true, y_pred, probs, text_col, n_examples=15):
    df = test_df.reset_index(drop=True).copy()
    df["true_label"] = [id2label[i] for i in y_true]
    df["pred_label"] = [id2label[i] for i in y_pred]
    df["confidence"] = probs.max(axis=1)
    wrong = df[df["true_label"] != df["pred_label"]].sort_values("confidence", ascending=False)
    print(f"\nTotal test examples: {len(df)} | Wrong: {len(wrong)} ({len(wrong)/len(df)*100:.1f}%)")
    print(f"\nTop {n_examples} most CONFIDENTLY WRONG predictions (hardest failures):")
    print(wrong[[text_col, "true_label", "pred_label", "confidence"]].head(n_examples).to_string(index=False))
    return wrong


def run_statistical_evaluation():
    print("\n" + "=" * 80)
    print("A. ADVANCED STATISTICAL EVALUATION")
    print("=" * 80)

    y_pred, y_true, probs = get_all_predictions(best_model, test_loader)

    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=id2label.values(), zero_division=0))

    print("--- Confusion Matrix ---")
    plot_confusion_matrix(y_true, y_pred, title="Fine-tuned Cross-Attention Model")

    print("\n--- Bootstrap 95% Confidence Intervals (1000 resamples) ---")
    acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_score)
    f1_mean, f1_lo, f1_hi = bootstrap_ci(y_true, y_pred, macro_f1)
    print(f"Accuracy : {acc_mean*100:.2f}%  [95% CI: {acc_lo*100:.2f}% - {acc_hi*100:.2f}%]")
    print(f"Macro F1 : {f1_mean*100:.2f}%  [95% CI: {f1_lo*100:.2f}% - {f1_hi*100:.2f}%]")

    print("\n--- Significance vs Majority-Class Baseline ---")
    majority_class = pd.Series(y_true).value_counts().idxmax()
    baseline_preds = np.full_like(y_true, majority_class)
    baseline_acc = accuracy_score(y_true, baseline_preds)
    print(f"Majority-class baseline accuracy: {baseline_acc*100:.2f}%")
    print(f"Your model accuracy            : {acc_mean*100:.2f}%")
    print(f"Improvement over baseline       : {(acc_mean-baseline_acc)*100:.2f} points")
    if acc_lo > baseline_acc:
        print("-> Model is statistically better than random/majority guessing (baseline falls outside 95% CI).")
    else:
        print("-> WARNING: baseline falls within the CI - investigate further.")

    print("\n--- Error Analysis ---")
    error_analysis(test_df, y_true, y_pred, probs, TEXT_COL)

    metrics_summary = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(macro_f1(y_true, y_pred)),
        "accuracy_ci95": [float(acc_lo), float(acc_hi)],
        "macro_f1_ci95": [float(f1_lo), float(f1_hi)],
    }
    with open("/kaggle/working/metrics.json", "w") as file:
        json.dump(metrics_summary, file, indent=2)
    print("\nSaved metrics to /kaggle/working/metrics.json :", metrics_summary)

    return y_true, y_pred, probs


# ================================================================================
# B. REAL-WORLD / UNSEEN DATA TESTING
# ================================================================================
PROBE_CASES = [
    ("The food was absolutely amazing, best meal ever!!", "Positive"),
    ("Terrible service, waited 2 hours for cold food.", "Negative"),
    ("It was okay, nothing special but not bad either.", "Neutral"),
    ("Oh great, ANOTHER overpriced dish with tiny portions. Just what I wanted.", "Negative"),
    ("Not bad at all, actually quite good.", "Positive"),
    ("The food was good but the service was really slow and rude.", "Negative"),
    ("Meh.", "Neutral"),
    ("Sedap sangat, best restaurant ever! Will definitely balik lagi.", "Positive"),
    ("Khana bahut acha tha par staff thoda rude tha.", "Negative"),
    ("The restaurant is located near the mall and has parking available.", "Neutral"),
    ("WORST EXPERIENCE OF MY LIFE. NEVER COMING BACK.", "Negative"),
    ("Decent place, average food, average price. Would go again if nearby.", "Neutral"),
]


def predict_texts(model, texts, bert_tok, roberta_tok, max_len=256):
    model.eval()
    preds, confidences = [], []
    with torch.no_grad():
        for text in texts:
            b = bert_tok(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
            r = roberta_tok(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
            logits = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                r["input_ids"].to(device),
                r["attention_mask"].to(device),
            )
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred = int(np.argmax(probs))
            preds.append(pred)
            confidences.append(float(probs[pred]))
    return preds, confidences


def run_real_world_testing():
    print("\n" + "=" * 80)
    print("B. REAL-WORLD / UNSEEN DATA TESTING (hand-crafted probe cases)")
    print("=" * 80)
    texts = [t for t, _ in PROBE_CASES]
    expected = [e for _, e in PROBE_CASES]
    preds, confs = predict_texts(best_model, texts, bert_tok, roberta_tok, MAX_LEN)

    rows = []
    correct = 0
    for text, exp, pred, conf in zip(texts, expected, preds, confs):
        pred_label = id2label[pred]
        ok = pred_label == exp
        correct += ok
        rows.append({
            "text": text[:60],
            "expected": exp,
            "predicted": pred_label,
            "confidence": f"{conf*100:.1f}%",
            "correct": "PASS" if ok else "FAIL",
        })
    probe_df = pd.DataFrame(rows)
    print(probe_df.to_string(index=False))
    print(f"\nProbe suite result: {correct}/{len(PROBE_CASES)} passed ({correct/len(PROBE_CASES)*100:.1f}%)")
    return probe_df


# ================================================================================
# C. PIPELINE / CODE TESTING (unit tests)
# ================================================================================
def run_unit_tests():
    print("\n" + "=" * 80)
    print("C. PIPELINE / CODE UNIT TESTS")
    print("=" * 80)
    passed, failed = 0, 0

    def check(name, condition):
        nonlocal passed, failed
        status = "PASS" if condition else "FAIL"
        if condition:
            passed += 1
        else:
            failed += 1
        print(f"[{status}] {name}")

    sample_text = "This restaurant is amazing."
    b = bert_tok(sample_text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    check("BERT tokenizer input_ids shape == (1, MAX_LEN)", b["input_ids"].shape == (1, MAX_LEN))
    check(
        "BERT tokenizer attention_mask is binary (0/1 only)",
        set(torch.unique(b["attention_mask"]).tolist()).issubset({0, 1}),
    )

    try:
        _ = bert_tok("", truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        check("Empty string input does not crash tokenizer", True)
    except Exception as e:
        check(f"Empty string input does not crash tokenizer ({e})", False)

    r = roberta_tok(sample_text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        logits = best_model(
            b["input_ids"].to(device),
            b["attention_mask"].to(device),
            r["input_ids"].to(device),
            r["attention_mask"].to(device),
        )
    check("Model output shape == (1, NUM_CLASSES=3)", tuple(logits.shape) == (1, 3))
    check("Model output contains no NaN/Inf", torch.isfinite(logits).all().item())

    probs = torch.softmax(logits, dim=1)
    check("Softmax probabilities sum to 1.0 (+/- 1e-4)", abs(probs.sum().item() - 1.0) < 1e-4)

    with torch.no_grad():
        logits2 = best_model(
            b["input_ids"].to(device),
            b["attention_mask"].to(device),
            r["input_ids"].to(device),
            r["attention_mask"].to(device),
        )
    check(
        "Same input produces identical prediction (deterministic in eval mode)",
        torch.argmax(logits, 1).item() == torch.argmax(logits2, 1).item(),
    )

    preds_single, _ = predict_texts(best_model, [sample_text], bert_tok, roberta_tok, MAX_LEN)
    preds_batch, _ = predict_texts(best_model, [sample_text, "Terrible food."], bert_tok, roberta_tok, MAX_LEN)
    check(
        "Prediction for a text is the same whether run alone or in a batch",
        preds_single[0] == preds_batch[0],
    )

    long_text = "very bad food " * 500
    try:
        _ = predict_texts(best_model, [long_text], bert_tok, roberta_tok, MAX_LEN)
        check("Very long input (>MAX_LEN tokens) truncates safely without crash", True)
    except Exception as e:
        check(f"Very long input truncates safely without crash ({e})", False)

    print(f"\nUnit test summary: {passed} passed, {failed} failed out of {passed+failed}")
    return passed, failed


y_true_test, y_pred_test, probs_test = run_statistical_evaluation()
run_real_world_testing()
run_unit_tests()

# 7. Déploiement — Production Readiness
Sauvegarde et rechargement du checkpoint (`best_model_finetuned.pt`), empreinte mémoire, benchmark de latence et wrapper d'inférence `predict_api()` avec journalisation JSONL.

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import torch

CHECKPOINT_PATH = "/kaggle/working/best_model_finetuned.pt"


def save_checkpoint(model, path=CHECKPOINT_PATH):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label_map": label2id,
            "max_len": MAX_LEN,
            "bert_name": BERT_NAME,
            "roberta_name": ROBERTA_NAME,
        },
        path,
    )
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"Checkpoint saved to {path} ({size_mb:.1f} MB)")


def load_checkpoint(path=CHECKPOINT_PATH):
    ckpt = torch.load(path, map_location=device)
    model = CrossAttentionFusionModel(NUM_CLASSES, freeze=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model


def predict_api(text, model=None, log_path="/kaggle/working/prediction_log.jsonl"):
    """Standalone inference function ready for FastAPI/Flask serving."""
    model = model or best_model
    start = time.time()
    preds, confs = predict_texts(model, [text], bert_tok, roberta_tok, MAX_LEN)
    latency_ms = (time.time() - start) * 1000
    result = {
        "text": text,
        "predicted_sentiment": id2label[preds[0]],
        "confidence": round(confs[0], 4),
        "latency_ms": round(latency_ms, 1),
        "timestamp": pd.Timestamp.now().isoformat(),
    }
    try:
        with open(log_path, "a") as f:
            f.write(json.dumps(result) + "\n")
    except Exception:
        pass
    return result


def benchmark_latency(model, texts, n_runs=50):
    _ = predict_texts(model, [texts[0]], bert_tok, roberta_tok, MAX_LEN)

    times = []
    for _ in range(n_runs):
        t0 = time.time()
        _ = predict_texts(model, [texts[0]], bert_tok, roberta_tok, MAX_LEN)
        times.append((time.time() - t0) * 1000)
    times = np.array(times)

    t0 = time.time()
    _ = predict_texts(model, texts, bert_tok, roberta_tok, MAX_LEN)
    batch_time = time.time() - t0
    throughput = len(texts) / batch_time

    print(f"Single-request latency over {n_runs} runs:")
    print(
        f"  mean: {times.mean():.1f} ms | p50: {np.percentile(times, 50):.1f} ms | "
        f"p95: {np.percentile(times, 95):.1f} ms | p99: {np.percentile(times, 99):.1f} ms"
    )
    print(f"Batch throughput ({len(texts)} texts): {throughput:.1f} texts/sec")
    if times.mean() > 500:
        print("-> WARNING: >500ms per request. Consider quantization or dynamic batching.")
    return times, throughput


def model_footprint(model):
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = n_params * 4 / (1024 ** 2)
    print(f"Total parameters     : {n_params:,}")
    print(f"Trainable parameters : {n_trainable:,}")
    print(f"Approx. size (fp32)  : {size_mb:.1f} MB")
    print(f"Device               : {device}")
    return n_params, size_mb


def run_production_readiness():
    print("\n" + "=" * 80)
    print("D. PRODUCTION READINESS")
    print("=" * 80)

    print("\n--- Save / Load Checkpoint ---")
    save_checkpoint(best_model)
    reloaded = load_checkpoint()
    sample = "This place has great ambience and friendly staff."
    p1, _ = predict_texts(best_model, [sample], bert_tok, roberta_tok, MAX_LEN)
    p2, _ = predict_texts(reloaded, [sample], bert_tok, roberta_tok, MAX_LEN)
    print(f"Reloaded checkpoint prediction matches original: {p1[0] == p2[0]}")

    print("\n--- Model Footprint ---")
    model_footprint(best_model)

    print("\n--- Latency Benchmark ---")
    sample_texts = test_df[TEXT_COL].astype(str).tolist()[:16]
    benchmark_latency(best_model, sample_texts)

    print("\n--- Example predict_api() call ---")
    example = predict_api("The dessert was disappointing but the mains were great.")
    print(json.dumps(example, indent=2))


run_production_readiness()

# Phase 5: Cross-Dataset / Out-of-Domain Testing
**Before running**: edit `NEW_DATASET_PATH`, `TEXT_COL_NEW`, `LABEL_COL_NEW`, `DATASET_NAME` below to point at whichever new dataset you want to test generalization on, and set `mapping` in the last line (`'stars'`, `'binary'`, or `'already_mapped'`).

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

NEW_DATASET_PATH = "/kaggle/input/datasets/choonkhonng/malaysia-restaurant-review-datasets/Malaysia Restaurant Review Datasets/data_cleaned/TripAdvisor_data_cleaned.csv"
TEXT_COL_NEW = "Review"
LABEL_COL_NEW = "Rating"
DATASET_NAME = "TripAdvisor Malaysia Restaurant Reviews"


def map_external_label(val, mapping):
    if mapping == "stars":
        v = float(val)
        if v >= 4:
            return "Positive"
        if v == 3:
            return "Neutral"
        return "Negative"
    if mapping == "binary":
        s = str(val).strip().lower()
        if s in {"1", "2", "positive", "pos", "true"}:
            return "Positive"
        if s in {"0", "-1", "negative", "neg", "false"}:
            return "Negative"
        return None
    if mapping == "already_mapped":
        s = str(val).strip().capitalize()
        return s if s in label2id else None
    raise ValueError(f"Unsupported mapping mode: {mapping}")


def evaluate_cross_dataset(path, text_col, label_col, dataset_name, mapping="stars", max_samples=3000, seed=42):
    print("\n" + "=" * 70)
    print(f"Loading {dataset_name}")
    print("=" * 70)

    df = pd.read_csv(path)[[text_col, label_col]].dropna().copy()
    df["text"] = df[text_col].astype(str).str.strip()
    df = df[df["text"].str.len() > 0].drop_duplicates(subset=["text"])
    df["sentiment"] = df[label_col].apply(lambda x: map_external_label(x, mapping))
    df = df.dropna(subset=["sentiment"])

    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=seed).reset_index(drop=True)

    df["label"] = df["sentiment"].map(label2id)
    print(f"Loaded {len(df)} usable rows. Class distribution:")
    print(df["sentiment"].value_counts())

    ext_loader = build_loader(df, shuffle=False)
    y_pred, y_true, _ = get_all_predictions(best_model, ext_loader)

    acc = accuracy_score(y_true, y_pred) * 100
    m_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) * 100
    w_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100
    print(f"\nAccuracy: {acc:.2f}% | Macro F1: {m_f1:.2f}% | Weighted F1: {w_f1:.2f}%")
    print(
        classification_report(
            y_true, y_pred, labels=[0, 1, 2], target_names=["Negative", "Neutral", "Positive"], zero_division=0
        )
    )
    plot_confusion_matrix(y_true, y_pred, title=dataset_name)


evaluate_cross_dataset(NEW_DATASET_PATH, TEXT_COL_NEW, LABEL_COL_NEW, DATASET_NAME, mapping="stars")